# PUSMILES Batch Encoder
### Polyurethane SMILES — Formulation to Machine-Readable String

**PUSMILES** encodes a polyurethane formulation as a single quantitative string:

```
*OCC(C)*@55{role=soft,Mn=3000} | *OC(=O)Nc1ccc(Cc2ccc(NC(=O)O*)cc2)cc1@40{role=hard} | OCCCCO@5{role=chain_ext}
```

Each component is its **SMILES structure** at its **weight percent**, with optional metadata. The full formulation is pipe-delimited and machine-readable.

**This notebook converts a batch CSV into one PUSMILES string per sample — no file uploads required except your CSV.**

---

### How it works

1. Cell 1 installs all dependencies and downloads the PUSMILES library automatically
2. Cell 2 lets you download a blank template CSV to fill in
3. Cell 3 uploads your completed CSV
4. Cells 4–6 resolve component names → SMILES using the library, PubChem, and Gemini AI as fallbacks
5. Cell 7 generates and downloads the output CSV

---
## Cell 1 — Setup

Downloads the PUSMILES library from GitHub, installs RDKit, and initializes the
AI resolution backend.

> **Just run this cell — nothing to configure or upload.**

In [ ]:
import subprocess, sys, io, os, re, json, warnings, textwrap, urllib.request, urllib.parse, urllib.error
import pandas as pd

# ── Install RDKit ─────────────────────────────────────────────────────────────
def _pip(pkg):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        stderr=subprocess.DEVNULL,
    )

try:
    from rdkit import Chem
except ImportError:
    print("Installing RDKit...")
    _pip("rdkit")
    from rdkit import Chem

# ── Download PUSMILES core library ───────────────────────────────────────────
# pusmiles.py is fetched once per session and cached locally.
PUSMILES_GITHUB_RAW = (
    "https://raw.githubusercontent.com/Loganz97/pusmiles/main/pusmiles.py"
    # ↑ Replace with your actual GitHub raw URL once the repo is created.
)

if not os.path.exists("pusmiles.py"):
    print("Downloading PUSMILES library...")
    try:
        urllib.request.urlretrieve(PUSMILES_GITHUB_RAW, "pusmiles.py")
        print("  Downloaded pusmiles.py from GitHub.")
    except Exception as e:
        raise RuntimeError(
            f"Could not download pusmiles.py: {e}\n"
            f"Check that the GitHub URL is correct: {PUSMILES_GITHUB_RAW}"
        )
else:
    print("pusmiles.py already present in session.")

from pusmiles import (
    PUSMILESBuilder, PUSMILESValidationError,
    POLYMER_REPEAT_UNITS, ADDITIVE_LIBRARY, BLEND_LIBRARY,
    is_blend_name, is_additive_name, pubchem_info,
    list_polymers, list_additives, list_blends,
)
print(f"PUSMILES library loaded.")
print(f"  Polymer library  : {len(list_polymers())} entries")
print(f"  Additive library : {len(list_additives())} entries")
print(f"  Blend library    : {len(list_blends())} entries")

# ── AI resolution backend (Gemini via google.colab.ai) ───────────────────────
# Used as final fallback when a column name is not in any library or PubChem.
# No API key required — google.colab.ai is available automatically in Colab.

try:
    from google.colab import ai as _colab_ai
    HAS_AI = True
    print("AI backend: google.colab.ai (Gemini) available.")
except ImportError:
    HAS_AI = False
    print("AI backend: not available (google.colab.ai not found — running outside Colab?).")
    print("Unknown component names will fall back to PubChem only.")

_AI_MODEL = "google/gemini-2.5-flash"

def _ai_generate(prompt, system):
    """Call Gemini 2.5 Flash via google.colab.ai."""
    if not HAS_AI:
        return None
    return _colab_ai.generate_text(f"{system}\n\n{prompt}", model=_AI_MODEL)

def _build_resolve_prompt(col_name, lib_names, blend_names, additive_names, roles):
    system = textwrap.dedent(f"""
        You are a polymer chemistry expert and PUSMILES specialist.

        A user has a CSV column named '{col_name}' in a polyurethane formulation.
        Identify what material this name refers to and return a JSON object.

        PUSMILES polymer library (prefer these names):
        {', '.join(lib_names)}

        PUSMILES blend library (commercial multi-component grades):
        {', '.join(blend_names)}

        PUSMILES additive library (prefer these names):
        {', '.join(additive_names)}

        Valid role strings:
        {', '.join(roles)}

        Return ONLY a JSON object with these keys:
        {{
          "library_name": "<exact library/blend/additive name if it matches, else null>",
          "smiles": "<valid SMILES string if no library match, else null>",
          "role": "<role string>",
          "reasoning": "<one sentence>"
        }}
        No markdown, no code fences, just the JSON object.
    """).strip()
    return system

print("\nSetup complete. Run the next cell to download the CSV template.")

---
## Cell 2 — Download a blank CSV template

Run this cell to download a pre-filled example CSV.

### CSV structure

| Column type | Example header | Value |
|---|---|---|
| **Sample ID** | `Sample_ID` | Any label — first column |
| **Component** | `PPG`, `MDI_urethane`, `CaCO3` | Weight percent (0–100) |
| **Molecular weight** | `PPG_Mn` | Mn for the component named before `_Mn` |
| **Functionality** | `PPG_fn` | Chain-end functionality for the named component |

### Rules
- Values are **weight percent**. Rows are auto-normalized if total is outside 99–101%.
- Enter **0 or leave blank** if a component is absent in a sample — it is skipped.
- Column names must match the PUSMILES library exactly (case-sensitive). See Cell 4 for resolution status.
- Unknown names are looked up on **PubChem**, then resolved by **Gemini AI** as a last resort.

### Recognized library names *(partial)*

| Category | Names |
|---|---|
| Polyether polyols | `PPG`, `PEG`, `PTMEG`, `PBO` |
| Polyester polyols | `PBA`, `PEA`, `PCL`, `PCDL` |
| MDI variants | `MDI_urethane`, `pMDI`, `MDI_modified` |
| TDI grades | `TDI_80`, `TDI_65`, `TDI_24_urethane`, `TDI_26_urethane` |
| Other isocyanates | `HDI_urethane`, `IPDI_urethane`, `H12MDI_urethane` |
| Chain extenders | `BDO`, `EG`, `HDO`, `BDO_urethane`, `EG_urethane`, `MOCA` |
| Crosslinkers | `glycerol`, `TMP`, `TEA`, `DEA` |
| Fillers | `CaCO3`, `BaSO4`, `silica`, `talc`, `glass_fiber` |
| Flame retardants | `TCPP`, `TEP`, `ATH`, `MEL`, `DMMP`, `DOPO` |
| Catalysts | `DABCO`, `DBTDL` |
| Blowing agents | `water`, `cyclopentane`, `n_pentane` |

In [ ]:
import csv

# Three representative PU formulations:
#   S001 — Basic flexible foam        : PPG-3000 triol / MDI / BDO chain extender
#   S002 — Filled flexible foam       : same + CaCO3 filler, DABCO catalyst, water blowing agent
#   S003 — Rigid flame-retarded foam  : short PPG triol / polymeric MDI / TEP flame retardant / cyclopentane

template = [
    # Headers
    # Sample ID  |── soft segment ──|  |─ hard segment ─|  |─ chain ext ─|  |────────────────── additives ──────────────────────|
    ["Sample_ID",  "PPG", "PPG_Mn", "PPG_fn",  "MDI_urethane",  "BDO",  "CaCO3",  "DABCO",  "water",  "pMDI",  "TEP",  "cyclopentane"],

    # S001: flexible foam — PPG-3000 triol (fn=3) / 4,4-MDI / BDO
    ["S001",        55,    3000,     3,           40,             5,      0,         0,        0,        0,       0,      0           ],

    # S002: filled flexible foam — same backbone + filler, catalyst, water blowing agent
    ["S002",        50,    3000,     3,           38,             7,      5,         1.5,      1,        0,       0,      0           ],

    # S003: rigid FR foam — short PPG triol / polymeric MDI / phosphate FR / cyclopentane BA
    ["S003",        30,    400,      3,           0,              0,      0,         0,        0,        50,      15,     5           ],
]

with open("pusmiles_template.csv", "w", newline="") as f:
    csv.writer(f).writerows(template)

try:
    from google.colab import files as _cf
    _cf.download("pusmiles_template.csv")
except ImportError:
    from IPython.display import display, FileLink
    display(FileLink("pusmiles_template.csv"))

print("Template saved: pusmiles_template.csv")
print("Fill in your data, then run Cell 3 to upload it.")

---
## Cell 3 — Upload your completed CSV

Click the file picker, select your filled-in CSV, and the encoder will load it.

In [ ]:
try:
    from google.colab import files as _cf
    print("Select your formulation CSV:")
    _up   = _cf.upload()
    _name = next(iter(_up))
    df    = pd.read_csv(io.BytesIO(_up[_name]))
except ImportError:
    # Local Jupyter fallback
    CSV_PATH = "pusmiles_template.csv"   # ← change to your file path
    df = pd.read_csv(CSV_PATH)

print(f"Loaded: {len(df)} sample rows, {len(df.columns)} columns")
df.head()

---
## Cell 4 — Classify columns

Identifies the sample ID column, component columns, and optional metadata columns
(`_Mn`, `_fn`). Review the output before proceeding — if something is misclassified,
rename the column in your CSV to match the rules in Cell 2.

In [ ]:
_META_SUFFIXES = {"_mn": "Mn", "_fn": "fn", "_mw": "Mw", "_cas": "cas"}
_ID_NAMES      = {"sample_id", "sample", "id", "name", "label", "run"}

def classify_columns(df):
    id_col    = None
    comp_cols = []
    meta_map  = {}
    skip      = set()

    for col in df.columns:
        low = col.lower()
        matched = False
        for suffix, key in _META_SUFFIXES.items():
            if low.endswith(suffix):
                meta_map[col] = (col[: -len(suffix)], key)
                skip.add(col)
                matched = True
                break
        if matched:
            continue
        if low in _ID_NAMES and id_col is None:
            id_col = col
            skip.add(col)
            continue
        if col not in skip:
            comp_cols.append(col)

    if id_col is None and comp_cols:
        first = comp_cols[0]
        if df[first].dtype == object:
            id_col    = first
            comp_cols = comp_cols[1:]

    return id_col, comp_cols, meta_map

ID_COL, COMP_COLS, META_MAP = classify_columns(df)

print(f"Sample ID column  : {ID_COL!r}")
print(f"\nComponent columns ({len(COMP_COLS)}):")
for col in COMP_COLS:
    print(f"  {col}")
if META_MAP:
    print(f"\nMetadata columns ({len(META_MAP)}):")
    for meta_col, (base, key) in META_MAP.items():
        print(f"  {meta_col:<20}  →  {key} for component '{base}'")

---
## Cell 5 — Resolve component names to SMILES

Each column header is matched in order:

| Step | Source | Handles |
|---|---|---|
| 1 | **Polymer library** | `PPG`, `MDI_urethane`, `BDO`, `PTMEG`, … |
| 2 | **Blend library** | `TDI_80`, `TDI_65`, `pMDI`, `MDI_modified` |
| 3 | **Additive library** | `CaCO3`, `DABCO`, `TCPP`, `water`, `BHT`, … |
| 4 | **NIH PubChem API** | Valid IUPAC names, common names, CAS numbers |
| 5 | **Gemini 2.5 Flash** | Trade names, abbreviations, informal names PubChem can't find |

Columns that survive all five steps without a match are flagged `FAILED`.

> **If a column shows `FAILED`:** try renaming it to an IUPAC name or CAS number
> so PubChem can find it, or use the exact SMILES string as the column header.

In [ ]:
_pubchem_cache = {}
_ai_cache      = {}

def _ai_resolve_col(col_name):
    """Ask Gemini 2.5 Flash to identify a compound name. Returns (id, role) or (None, None)."""
    if not HAS_AI or col_name in _ai_cache:
        return _ai_cache.get(col_name, (None, None))

    lib_names     = sorted(POLYMER_REPEAT_UNITS.keys())
    blend_names   = sorted(BLEND_LIBRARY.keys())
    additive_names = sorted(ADDITIVE_LIBRARY.keys())
    from pusmiles import ADDITIVE_ROLES
    roles         = sorted(ADDITIVE_ROLES)

    system = _build_resolve_prompt(col_name, lib_names, blend_names, additive_names, roles)

    try:
        raw = _ai_generate(system, "")   # full prompt is in system arg
        if raw is None:
            _ai_cache[col_name] = (None, None)
            return (None, None)
        raw = raw.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-z]*\n?", "", raw)
            raw = re.sub(r"\n?```$",       "", raw).strip()
        result   = json.loads(raw)
        lib_name = result.get("library_name")
        smiles   = result.get("smiles")
        role     = result.get("role", "additive")
        out      = (lib_name or smiles, role)
    except Exception as e:
        print(f"    Gemini resolution failed for '{col_name}': {e}")
        out = (None, None)

    _ai_cache[col_name] = out
    return out


def resolve_col(col_name):
    """Returns (identifier, role, use_additive_lib). Resolution: library → PubChem → Gemini."""
    # 1. Polymer / blend library
    if col_name in POLYMER_REPEAT_UNITS or is_blend_name(col_name):
        return col_name, None, False
    # 2. Additive library
    if is_additive_name(col_name):
        return col_name, ADDITIVE_LIBRARY[col_name]["role"], True
    # 3. Raw SMILES heuristic
    if re.search(r"[=#@\[\]/%]", col_name):
        return col_name, "additive", False
    # 4. PubChem
    if col_name not in _pubchem_cache:
        try:
            _pubchem_cache[col_name] = pubchem_info(col_name)
        except Exception:
            _pubchem_cache[col_name] = None
    if _pubchem_cache.get(col_name):
        return _pubchem_cache[col_name]["smiles"], "additive", False
    # 5. Gemini AI
    ai_id, ai_role = _ai_resolve_col(col_name)
    if ai_id:
        if is_additive_name(ai_id):
            return ai_id, ai_role, True
        if ai_id in POLYMER_REPEAT_UNITS or is_blend_name(ai_id):
            return ai_id, ai_role, False
        return ai_id, ai_role, False
    return None, None, False


print(f"{'Column':<25}  {'Source':<14}  {'Resolved to'}")
print("-" * 85)
COL_REGISTRY = {}
for col in COMP_COLS:
    smiles, role, use_add = resolve_col(col)
    COL_REGISTRY[col] = (smiles, role, use_add)
    if smiles is None:
        source = "FAILED"
    elif use_add:
        source = "additive-lib"
    elif smiles == col:
        source = "library"
    elif col in _ai_cache and _ai_cache[col][0]:
        source = "Gemini AI"
    elif _pubchem_cache.get(col):
        source = "PubChem"
    else:
        source = "raw SMILES"
    label = (smiles or "not resolved")[:55]
    print(f"  {col:<23}  [{source:<12}]  {label}")

---
## Cell 6 — Generate PUSMILES strings

Assembles one PUSMILES string per sample row. Zero-weight components are skipped.
Weights outside 99–101% are normalized automatically. Errors are recorded per row
and visible in the output CSV.

In [ ]:
PUSMILES_COL = "PUSMILES"
VALID_COL    = "valid"
ERRORS_COL   = "errors"
results      = []

for row_idx, row in df.iterrows():
    sample_id  = row[ID_COL] if ID_COL else row_idx
    row_errors = []

    row_meta = {}
    for meta_col, (base, key) in META_MAP.items():
        val = row.get(meta_col)
        if pd.notna(val) and str(val).strip() not in ("", "0"):
            row_meta.setdefault(base, {})[key] = val

    builder = PUSMILESBuilder()

    for col in COMP_COLS:
        raw_wt = row.get(col, 0)
        if pd.isna(raw_wt) or str(raw_wt).strip() in ("", "0") or float(raw_wt) == 0:
            continue
        wt = float(raw_wt)

        smiles, role, use_add = COL_REGISTRY[col]
        if smiles is None:
            row_errors.append(f"unresolved: '{col}'")
            continue

        meta = dict(row_meta.get(col, {}))
        if role:
            meta["role"] = role

        try:
            with warnings.catch_warnings(record=True):
                warnings.simplefilter("always")
                if use_add:
                    builder.add_additive(col, wt, **meta)
                else:
                    builder.add(smiles, wt, **meta)
        except Exception as e:
            row_errors.append(f"'{col}': {e}")

    pusmiles_str = ""
    valid        = False

    if not builder._components:
        row_errors.append("no components — all weights zero or unresolved")
    else:
        try:
            with warnings.catch_warnings(record=True):
                warnings.simplefilter("always")
                q = builder.build()
            if not (99 <= q.total_weight() <= 101):
                q = q.normalize()
            pusmiles_str = q.to_string()
            valid        = q.validate().is_valid
        except PUSMILESValidationError as e:
            row_errors += [str(err) for err in e.result.errors]

    results.append({
        ID_COL or "index": sample_id,
        PUSMILES_COL:      pusmiles_str,
        VALID_COL:         valid,
        ERRORS_COL:        "; ".join(row_errors) if row_errors else "",
    })

df_out = pd.DataFrame(results)
n_ok   = int(df_out[VALID_COL].sum())
n_fail = len(df_out) - n_ok
print(f"Generated PUSMILES for {len(df_out)} samples: {n_ok} valid, {n_fail} with errors.")
df_out[[ID_COL or "index", VALID_COL, PUSMILES_COL]].head(10)

---
## Cell 7 — Download output CSV

Saves `pusmiles_output.csv` and triggers a browser download.

| Output column | Description |
|---|---|
| `Sample_ID` | Copied from your input |
| `PUSMILES` | Encoded formulation string |
| `valid` | `True` if the PUSMILES passed all structural checks |
| `errors` | Issues for that row, if any |

In [ ]:
OUT_FILE = "pusmiles_output.csv"
df_out.to_csv(OUT_FILE, index=False)

try:
    from google.colab import files as _cf
    _cf.download(OUT_FILE)
except ImportError:
    from IPython.display import display, FileLink
    display(FileLink(OUT_FILE))

print(f"Saved {len(df_out)} rows to {OUT_FILE}")

flagged = df_out[df_out[ERRORS_COL] != ""]
if not flagged.empty:
    print(f"\n{len(flagged)} row(s) with errors:")
    for _, r in flagged.iterrows():
        print(f"  [{r[ID_COL or 'index']}]  {r[ERRORS_COL]}")

---
## Cell 8 — Inspect a single sample *(optional)*

Parses a PUSMILES string back into its components. Useful for spot-checking
before passing output to a model. Change `SAMPLE_ID` to any value from your output.

In [ ]:
from pusmiles import PUSMILES

SAMPLE_ID = df_out.iloc[0][ID_COL or "index"]   # ← change to any Sample_ID value

row = df_out[df_out[ID_COL or "index"] == SAMPLE_ID]
if row.empty:
    print(f"Sample {SAMPLE_ID!r} not found.")
else:
    pus = row.iloc[0][PUSMILES_COL]
    if not pus:
        print(f"No PUSMILES for {SAMPLE_ID!r}. See errors column.")
    else:
        q = PUSMILES.parse(pus)
        print(f"Sample   : {SAMPLE_ID}")
        print(f"PUSMILES : {pus[:110]}{'...' if len(pus) > 110 else ''}")
        print()
        print(f"  {'#':<3} {'Role':<18} {'Component':<42} {'wt%':>6}  Metadata")
        print(f"  {'-' * 80}")
        for i, c in enumerate(q.components):
            label    = c.metadata.get("name") or c.smiles
            label    = (label[:41] + "...") if len(label) > 42 else label
            role_str = c.role or ""
            meta_str = ", ".join(
                f"{k}={v}" for k, v in c.metadata.items()
                if k not in ("role", "name")
            )
            print(f"  {i:<3} {role_str:<18} {label:<42} {c.weight_pct:>6.2f}  {meta_str}")
        print(f"  {'-' * 80}")
        valid = q.validate()
        print(f"  Total: {q.total_weight():.2f} wt%  |  Validation: {'PASS' if valid.is_valid else 'FAIL'}")